# TOMATO QUALITY & RIPENESS - UNIFIED 5-CLASS MODEL

**Version: Multi-GPU Optimized - January 28, 2026**

**Setup:**
- Kaggle Verified Account
- T4*2 (2 Tesla T4 GPUs = ~30GB total VRAM)
- 45 hours notebook time available

**Training Configuration:**
- Multi-GPU training (device=[0,1])
- Batch size: 32 (64 total across 2 GPUs)
- YOLOv8 Medium model
- 640px resolution
- 150 epochs with early stopping (patience=25)
- Hard negative optimization (Mosaic 1.0, Erasing 0.4)

**Classification Logic:**
- ACCEPT: Green, Breaker, Turning, Ripe
- REJECT: Defective tomatoes

---

**Execution Steps:**
1. Install & Import (Cell 2)
2. Load Dataset (Cell 4)
3. Load Model (Cell 6)
4. Training Configuration (Cell 8)
5. Execute Training (Cell 11) - 3-4 hours
6. Extract Results (Cell 13)
7. Analyze Results (Cells 14-30)

## CELL 2: Install & Import Libraries

In [ ]:
%pip install -q ultralytics opencv-python torch torchvision matplotlib pillow pandas

import os
import sys
import json
import shutil
import glob
from pathlib import Path
from datetime import datetime
import yaml
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import pandas as pd

from ultralytics import YOLO
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

## CELL 4: Load 5-Class Dataset

In [ ]:
data_path = None

if os.path.exists('/kaggle/input'):
    datasets = os.listdir('/kaggle/input')
    if datasets:
        data_path = os.path.join('/kaggle/input', datasets[0])
else:
    for path in ['data', '../data', '.']:
        if os.path.exists(os.path.join(path, 'data.yaml')):
            data_path = path
            break

if data_path is None:
    raise FileNotFoundError("Dataset not found.")

print(f"Dataset path: {data_path}")

with open(os.path.join(data_path, 'data.yaml'), 'r') as f:
    data_yaml = yaml.safe_load(f)
    names = data_yaml['names']
    
    if isinstance(names, dict):
        classes_list = list(names.values())
    else:
        classes_list = names
    
    print(f"Classes: {classes_list}")
    print(f"Total classes: {len(classes_list)}")

## CELL 6: Load YOLOv8 Model

In [ ]:
model = YOLO('yolov8m.pt')
print("Model loaded.")

## CELL 8: Training Configuration

In [ ]:
training_config = {
    'model': 'yolov8m.pt',
    'data': os.path.join(data_path, 'data.yaml'),
    'epochs': 150,
    'imgsz': 640,
    'batch': 32,
    'device': [0, 1],
    'workers': 4,
    'cache': 'disk',
    'patience': 25,
    'save': True,
    'save_period': 5,
    'lr0': 0.01,
    'lrf': 0.01,
    'momentum': 0.937,
    'weight_decay': 0.0005,
    'warmup_epochs': 3.0,
    'warmup_momentum': 0.8,
    'box': 7.5,
    'cls': 0.5,
    'dfl': 1.5,
    'hsv_h': 0.015,
    'hsv_s': 0.7,
    'hsv_v': 0.4,
    'degrees': 15,
    'translate': 0.1,
    'scale': 0.5,
    'flipud': 0.0,
    'fliplr': 0.5,
    'mosaic': 1.0,
    'mixup': 0.15,
    'erasing': 0.4,
    'perspective': 0.0,
    'shear': 0.0,
}

print("Training configuration ready:")
print(f"  Epochs: {training_config['epochs']}")
print(f"  Early stopping patience: {training_config['patience']}")
print(f"  Batch size: {training_config['batch']}")
print(f"  GPUs: {training_config['device']}")

## CELL 11: START TRAINING

Estimated duration: 3-4 hours

In [ ]:
start_time = datetime.now()
print(f"Training started: {start_time.strftime('%Y-%m-%d %H:%M:%S')}")

try:
    results = model.train(
        data=training_config['data'],
        epochs=training_config['epochs'],
        imgsz=training_config['imgsz'],
        batch=training_config['batch'],
        device=training_config['device'],
        workers=training_config['workers'],
        cache=training_config['cache'],
        patience=training_config['patience'],
        save=training_config['save'],
        save_period=training_config['save_period'],
        lr0=training_config['lr0'],
        lrf=training_config['lrf'],
        momentum=training_config['momentum'],
        weight_decay=training_config['weight_decay'],
        warmup_epochs=training_config['warmup_epochs'],
        warmup_momentum=training_config['warmup_momentum'],
        box=training_config['box'],
        cls=training_config['cls'],
        dfl=training_config['dfl'],
        hsv_h=training_config['hsv_h'],
        hsv_s=training_config['hsv_s'],
        hsv_v=training_config['hsv_v'],
        degrees=training_config['degrees'],
        translate=training_config['translate'],
        scale=training_config['scale'],
        flipud=training_config['flipud'],
        fliplr=training_config['fliplr'],
        mosaic=training_config['mosaic'],
        mixup=training_config['mixup'],
        erasing=training_config['erasing'],
        project='/kaggle/working',
        name='tomato_5class_model',
        exist_ok=True,
        verbose=True
    )
    
    end_time = datetime.now()
    duration = end_time - start_time
    print(f"Training completed - Duration: {duration}")
    
except Exception as e:
    print(f"Error: {str(e)}")
    raise

## CELL 13: COMPREHENSIVE RESULTS EXTRACTION & ORGANIZATION

In [ ]:
checkpoint_dir = '/kaggle/working/tomato_5class_model'
results_dir = os.path.join(checkpoint_dir, 'results')

print("Extracting and organizing all training results...")

if os.path.exists(checkpoint_dir):
    os.makedirs(results_dir, exist_ok=True)
    total_extracted_size = 0
    
    print("Extracting training graphs...")
    graphs_dir = os.path.join(results_dir, 'graphs')
    os.makedirs(graphs_dir, exist_ok=True)
    
    graph_files = glob.glob(os.path.join(checkpoint_dir, '*.png')) + glob.glob(os.path.join(checkpoint_dir, '*.jpg'))
    
    extracted_count = 0
    for graph_file in graph_files:
        filename = os.path.basename(graph_file)
        dest_path = os.path.join(graphs_dir, filename)
        shutil.copy2(graph_file, dest_path)
        extracted_count += 1
    
    print(f"Graphs extracted: {extracted_count}")
    
    print("Copying metrics and configuration...")
    csv_src = os.path.join(checkpoint_dir, 'results.csv')
    if os.path.exists(csv_src):
        shutil.copy2(csv_src, os.path.join(results_dir, 'results.csv'))
    
    yaml_src = os.path.join(checkpoint_dir, 'args.yaml')
    if os.path.exists(yaml_src):
        shutil.copy2(yaml_src, os.path.join(results_dir, 'args.yaml'))
    
    print("Copying model weights...")
    weights_dir = os.path.join(results_dir, 'weights')
    os.makedirs(weights_dir, exist_ok=True)
    
    weights_source = os.path.join(checkpoint_dir, 'weights')
    if os.path.exists(weights_source):
        for weight_file in os.listdir(weights_source):
            src_path = os.path.join(weights_source, weight_file)
            if os.path.isfile(src_path):
                shutil.copy2(src_path, os.path.join(weights_dir, weight_file))
    
    print(f"Extraction complete - Location: {results_dir}")
else:
    print(f"Error: Training directory not found")

## CELL 13B: Find & List All Generated Graphs

In [ ]:
checkpoint_dir = '/kaggle/working/tomato_5class_model'

print("Scanning training directory for all graphs...")
print(f"Location: {checkpoint_dir}\n")

all_files = []
if os.path.exists(checkpoint_dir):
    for root, dirs, files in os.walk(checkpoint_dir):
        for file in files:
            full_path = os.path.join(root, file)
            rel_path = os.path.relpath(full_path, checkpoint_dir)
            size_mb = os.path.getsize(full_path) / (1024 * 1024)
            all_files.append({
                'name': file,
                'path': full_path,
                'rel_path': rel_path,
                'size_mb': size_mb,
                'type': file.split('.')[-1]
            })

graph_extensions = ['png', 'jpg', 'jpeg']
graphs = [f for f in all_files if f['type'].lower() in graph_extensions]

print(f"Total files found: {len(all_files)}")
print(f"Total graphs found: {len(graphs)}\n")

if graphs:
    print("Available graphs for download:")
    print("-" * 80)
    for idx, graph in enumerate(graphs, 1):
        print(f"{idx}. {graph['name']:<40} ({graph['size_mb']:.2f} MB)")
    print("-" * 80)
else:
    print("No graphs found yet. Training may still be in progress.")

print(f"\nOther files: {len([f for f in all_files if f['type'].lower() not in graph_extensions])}")


## CELL 13C: Prepare All Graphs for Download

In [ ]:
checkpoint_dir = '/kaggle/working/tomato_5class_model'
download_folder = '/kaggle/working/TOMATO_TRAINING_GRAPHS'

os.makedirs(download_folder, exist_ok=True)

print("Copying all graphs to download folder...")
print(f"Download location: {download_folder}\n")

copy_count = 0
failed = []

for graph in graphs:
    try:
        dest_path = os.path.join(download_folder, graph['name'])
        shutil.copy2(graph['path'], dest_path)
        copy_count += 1
        print(f"Copied: {graph['name']}")
    except Exception as e:
        failed.append((graph['name'], str(e)))
        print(f"Failed: {graph['name']} - {e}")

print(f"\nResults:")
print(f"  Successfully copied: {copy_count} graphs")
print(f"  Failed: {len(failed)} graphs")

if copy_count > 0:
    print(f"\n{copy_count} graphs are now ready for download in:")
    print(f"  {download_folder}")


## CELL 13D: Verify Download Folder Contents

In [ ]:
download_folder = '/kaggle/working/TOMATO_TRAINING_GRAPHS'

print("Verifying download folder contents...\n")

if os.path.exists(download_folder):
    files = os.listdir(download_folder)
    print(f"Download Folder: {download_folder}")
    print(f"Total files ready: {len(files)}\n")
    
    if files:
        print("Files ready for download:")
        print("-" * 80)
        total_size = 0
        for idx, filename in enumerate(sorted(files), 1):
            filepath = os.path.join(download_folder, filename)
            size_mb = os.path.getsize(filepath) / (1024 * 1024)
            total_size += size_mb
            print(f"{idx}. {filename:<50} ({size_mb:>6.2f} MB)")
        print("-" * 80)
        print(f"Total size: {total_size:.2f} MB")
        print(f"\nStatus: Ready for download in Kaggle Output folder")
    else:
        print("Folder is empty - graphs may not have been generated yet")
else:
    print(f"Error: Download folder not found at {download_folder}")


## CELL 13E: Organize Output Folder Structure

In [ ]:
checkpoint_dir = '/kaggle/working/tomato_5class_model'
output_root = '/kaggle/working/TOMATO_MODEL_OUTPUTS'

print("Creating organized output folder structure...\n")

folders = {
    '1_MODELS': os.path.join(output_root, '1_MODELS'),
    '2_TRAINING_GRAPHS': os.path.join(output_root, '2_TRAINING_GRAPHS'),
    '3_VALIDATION_RESULTS': os.path.join(output_root, '3_VALIDATION_RESULTS'),
    '4_METRICS': os.path.join(output_root, '4_METRICS'),
}

for folder_name, folder_path in folders.items():
    os.makedirs(folder_path, exist_ok=True)
    print(f"Created: {folder_name}")

print("\n" + "="*80)
print("Organizing files into folders...")
print("="*80 + "\n")

copy_operations = {
    '1_MODELS': [
        ('weights/best.pt', 'best.pt'),
        ('weights/last.pt', 'last.pt'),
    ],
    '2_TRAINING_GRAPHS': [
        ('results.png', 'results.png'),
        ('confusion_matrix.png', 'confusion_matrix.png'),
        ('BoxPR_curve.png', 'BoxPR_curve.png'),
        ('BoxP_curve.png', 'BoxP_curve.png'),
        ('BoxR_curve.png', 'BoxR_curve.png'),
        ('BoxF1_curve.png', 'BoxF1_curve.png'),
        ('labels.jpg', 'labels.jpg'),
    ],
    '3_VALIDATION_RESULTS': [
        ('train_batch.jpg', 'train_batch.jpg'),
        ('val_batch_labels.jpg', 'val_batch_labels.jpg'),
        ('val_batch_pred.jpg', 'val_batch_pred.jpg'),
    ],
    '4_METRICS': [
        ('results.csv', 'results.csv'),
        ('args.yaml', 'args.yaml'),
    ],
}

total_copied = 0
for folder_key, file_list in copy_operations.items():
    dest_folder = folders[folder_key]
    for src_file, dest_file in file_list:
        src_path = os.path.join(checkpoint_dir, src_file)
        dest_path = os.path.join(dest_folder, dest_file)
        
        if os.path.exists(src_path):
            try:
                shutil.copy2(src_path, dest_path)
                print(f"✓ {folder_key:<25} <- {src_file}")
                total_copied += 1
            except Exception as e:
                print(f"✗ {folder_key:<25} <- {src_file} (Error: {e})")
        else:
            print(f"- {folder_key:<25} <- {src_file} (Not found)")

print(f"\n{'='*80}")
print(f"Total files organized: {total_copied}")
print(f"Output location: {output_root}")
print(f"{'='*80}\n")


## CELL 13F: Verify Organized Structure

In [ ]:
output_root = '/kaggle/working/TOMATO_MODEL_OUTPUTS'

print("FINAL FOLDER STRUCTURE:\n")
print(f"Location: {output_root}\n")
print("="*80)

def print_tree(path, prefix="", is_last=True):
    if not os.path.exists(path):
        return
    
    items = sorted(os.listdir(path))
    dirs = [d for d in items if os.path.isdir(os.path.join(path, d))]
    files = [f for f in items if os.path.isfile(os.path.join(path, f))]
    
    for i, d in enumerate(dirs):
        is_last_dir = (i == len(dirs) - 1) and len(files) == 0
        print(f"{prefix}{'└── ' if is_last_dir else '├── '}{d}/")
        
        new_prefix = prefix + ("    " if is_last_dir else "│   ")
        print_tree(os.path.join(path, d), new_prefix, is_last_dir)
    
    for i, f in enumerate(files):
        is_last_file = i == len(files) - 1
        size = os.path.getsize(os.path.join(path, f)) / (1024 * 1024)
        print(f"{prefix}{'└── ' if is_last_file else '├── '}{f:<40} ({size:>6.2f} MB)")

print_tree(output_root)
print("="*80)

total_size = sum(os.path.getsize(os.path.join(root, f)) 
                for root, dirs, files in os.walk(output_root) 
                for f in files) / (1024 * 1024)

print(f"\nTotal output size: {total_size:.2f} MB")
print(f"Status: Ready for download in Kaggle Output folder")


## CELL 14: Extract & Display Final Metrics Summary

In [ ]:
training_dir = os.path.join('/kaggle/working/tomato_5class_model')
csv_file = os.path.join(training_dir, 'results.csv')

if os.path.exists(csv_file):
    df = pd.read_csv(csv_file)
    
    print("Training Summary Statistics")
    print("="*60)
    
    final_row = df.iloc[-1]
    
    print(f"Total epochs trained: {len(df)}")
    print("\nFinal Performance:")
    if 'metrics/mAP50' in df.columns:
        print(f"  mAP50: {final_row['metrics/mAP50']:.4f}")
    if 'metrics/precision' in df.columns:
        print(f"  Precision: {final_row['metrics/precision']:.4f}")
    if 'metrics/recall' in df.columns:
        print(f"  Recall: {final_row['metrics/recall']:.4f}")
else:
    print("Results CSV not found.")

## CELL 16: Training Curves Visualization

In [ ]:
results_png = os.path.join(training_dir, 'results.png')

if os.path.exists(results_png):
    img = Image.open(results_png)
    plt.figure(figsize=(16, 8))
    plt.imshow(img)
    plt.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print(f"Results PNG not found")

## CELL 18: Confusion Matrix Analysis

In [ ]:
confusion_matrix_png = os.path.join(training_dir, 'confusion_matrix.png')

if os.path.exists(confusion_matrix_png):
    img = Image.open(confusion_matrix_png)
    plt.figure(figsize=(12, 10))
    plt.imshow(img)
    plt.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print(f"Confusion matrix not found")

## CELL 20: Detailed Metrics Table

In [ ]:
if os.path.exists(csv_file):
    df = pd.read_csv(csv_file)
    
    print("Last 10 Epochs - Key Metrics")
    
    display_cols = [col for col in df.columns if any(x in col for x in ['epoch', 'mAP', 'precision', 'recall', 'loss'])]
    display_df = df[display_cols].tail(10).round(4)
    
    print(display_df.to_string(index=False))

## CELL 22: Prepare & Download Results

In [ ]:
print("Preparing results for download.")

results_mapping = {
    'Best Model Weights': 'weights/best.pt',
    'Last Model Weights': 'weights/last.pt',
    'Training Results CSV': 'results.csv',
    'Confusion Matrix': 'confusion_matrix.png',
    'Training Curves': 'results.png',
}

download_dir = '/kaggle/working/TOMATO_MODEL_RESULTS'
os.makedirs(download_dir, exist_ok=True)

copied_count = 0
for name, rel_path in results_mapping.items():
    full_path = os.path.join(training_dir, rel_path)
    if os.path.exists(full_path):
        dest_path = os.path.join(download_dir, os.path.basename(rel_path))
        shutil.copy2(full_path, dest_path)
        copied_count += 1

print(f"Files prepared: {copied_count}/{len(results_mapping)}")

## CELL 24: Load & Verify Best Model

In [ ]:
best_model_path = os.path.join(training_dir, 'weights/best.pt')

if os.path.exists(best_model_path):
    best_model = YOLO(best_model_path)
    print("Best model loaded.")
else:
    print(f"Error: Best model not found")

## CELL 26: All Training Graphs & Visualizations

In [ ]:
print("Displaying training graphs...")

graphs = [
    ("Training Curves", 'results.png'),
    ("Confusion Matrix", 'confusion_matrix.png'),
    ("Precision-Recall Curve", 'BoxPR_curve.png'),
    ("Precision Curve", 'BoxP_curve.png'),
    ("Recall Curve", 'BoxR_curve.png'),
    ("F1 Curve", 'BoxF1_curve.png'),
    ("Training Batch Examples", 'train_batch.jpg'),
    ("Validation Batch - Ground Truth", 'val_batch_labels.jpg'),
    ("Validation Batch - Model Predictions", 'val_batch_pred.jpg'),
    ("Class Distribution", 'labels.jpg'),
]

for idx, (title, filename) in enumerate(graphs, 1):
    graph_path = os.path.join(training_dir, filename)
    if os.path.exists(graph_path):
        print(f"\n{idx}. {title}")
        img = Image.open(graph_path)
        plt.figure(figsize=(14, 10))
        plt.imshow(img)
        plt.axis('off')
        plt.tight_layout()
        plt.show()

## CELL 28: Performance Analysis

In [ ]:
print("Performance Analysis:")

if os.path.exists(csv_file):
    df = pd.read_csv(csv_file)
    
    print(f"Total epochs: {len(df)}")
    
    if 'metrics/mAP50' in df.columns:
        best_map_idx = df['metrics/mAP50'].idxmax()
        best_map = df.loc[best_map_idx, 'metrics/mAP50']
        print(f"Best mAP50: {best_map:.4f} (epoch {int(df.loc[best_map_idx, 'epoch'])+1})")
    
    if 'metrics/precision' in df.columns:
        best_prec_idx = df['metrics/precision'].idxmax()
        best_prec = df.loc[best_prec_idx, 'metrics/precision']
        print(f"Best Precision: {best_prec:.4f} (epoch {int(df.loc[best_prec_idx, 'epoch'])+1})")
    
    if 'metrics/recall' in df.columns:
        best_recall_idx = df['metrics/recall'].idxmax()
        best_recall = df.loc[best_recall_idx, 'metrics/recall']
        print(f"Best Recall: {best_recall:.4f} (epoch {int(df.loc[best_recall_idx, 'epoch'])+1})")

## CELL 30: Final Verification

In [ ]:
print("Verification Checklist:")

checks = []
checkpoint = '/kaggle/working/tomato_5class_model'

check1 = os.path.exists(checkpoint)
checks.append(("PASS" if check1 else "FAIL", "Training folder exists"))

best_weights = os.path.join(checkpoint, 'weights/best.pt')
check2 = os.path.exists(best_weights)
checks.append(("PASS" if check2 else "FAIL", "Best model weights"))

cm = os.path.join(checkpoint, 'confusion_matrix.png')
check3 = os.path.exists(cm)
checks.append(("PASS" if check3 else "FAIL", "Confusion matrix"))

csv = os.path.join(checkpoint, 'results.csv')
check4 = os.path.exists(csv)
checks.append(("PASS" if check4 else "FAIL", "Training metrics"))

results_dir = os.path.join(checkpoint, 'results')
check5 = os.path.exists(results_dir)
checks.append(("PASS" if check5 else "FAIL", "Results folder"))

download_dir = '/kaggle/working/TOMATO_MODEL_RESULTS'
check6 = os.path.exists(download_dir)
checks.append(("PASS" if check6 else "FAIL", "Download folder"))

for status, description in checks:
    print(f"{status}: {description}")

passed = sum(1 for s, _ in checks if s == "PASS")
total = len(checks)
print(f"\nResults: {passed}/{total} checks passed")